# 실제 테스트 코드


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#라이브러리 임포트
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
import glob
import zipfile
import os, json, numpy as np
from PIL import Image
import torchvision.models as models
from torchvision import transforms

In [6]:
#zip 파일 압축 해제
zip_path = '/content/drive/MyDrive/real_skeleton_images.zip'
extract_path = '/content/real_skeleton_images'

# 압축 해제
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# 압축 해제된 경로 확인
os.listdir(extract_path)


['skeleton_images']

In [7]:
class CNNLSTMClassifier(nn.Module):
    def __init__(self, hidden_dim=256, lstm_layers=2, dropout_rate=0.3):
        super(CNNLSTMClassifier, self).__init__()

        base_model = models.resnet18(weights='DEFAULT')
        base_model.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.cnn_backbone = nn.Sequential(*list(base_model.children())[:-1])
        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout_rate if lstm_layers > 1 else 0.0
        )

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(-1, C, H, W)
        features = self.cnn_backbone(x).view(B, T, -1)

        lstm_out, _ = self.lstm(features)
        last_hidden = lstm_out[:, -1, :]

        out = self.fc(last_hidden)
        return out.squeeze(-1)


In [ ]:

# 원하는 자세 경로 입력
sequence_folder = "/content/real_skeleton_images/skeleton_images/HipCircle/HipCircle_good"

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

image_files = sorted([
    f for f in os.listdir(sequence_folder)
    if f.startswith("frame_") and f.endswith(".png")
])

images = []
for img_name in image_files:
    img_path = os.path.join(sequence_folder, img_name)
    img = Image.open(img_path).convert("RGB")
    images.append(transform(img).numpy())

sequence_tensor = torch.tensor(np.stack(images)).unsqueeze(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sequence_tensor = sequence_tensor.to(device)

model = CNNLSTMClassifier(hidden_dim=256, lstm_layers=2, dropout_rate=0.5).to(device)
model.load_state_dict(torch.load(
    "/content/drive/MyDrive/Hip Circles_cnn_lstm_model.pth",
    map_location=device
))
model.eval()

with torch.no_grad():
    logits = model(sequence_tensor)
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).int()

#  결과 출력
print("동작 정확도 예측 결과:")
print(f"결과는 ～ {'옳바른 자세!' if preds.item() == 1 else '틀린 자세 ㅠ'}  ")


동작 정확도 예측 결과:
결과는 ～ 옳바른 자세!  
